In [ ]:
# --- Path setup ---
import sys
import os
import math
import time
import contextlib
import io
import warnings
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "max_k_cut").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

# --- Scientific / data ---
import networkx as nx
import numpy as np
import scipy.io

# --- Visualization ---
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams["figure.dpi"] = 1000
mpl.rcParams["savefig.dpi"] = 1000

# --- Qiskit ecosystem ---
import qiskit
import qiskit_aer
import qiskit_algorithms
import qiskit_optimization
from qiskit.circuit import Parameter
from qiskit import QuantumCircuit
from qiskit.primitives import Sampler, BackendSampler
from qiskit.circuit.library import QAOAAnsatz, RYGate, XGate, CXGate
from qiskit.visualization import plot_histogram, plot_state_city, plot_state_qsphere, plot_bloch_multivector, plot_distribution
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeBrisbane


from qiskit_algorithms import QAOA, SamplingVQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA

from qiskit_optimization.algorithms import MinimumEigenOptimizer, SolutionSample, OptimizationResultStatus
from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import LinearEqualityToPenalty, LinearInequalityToPenalty, QuadraticProgramToQubo
from qiskit_optimization.translators import from_docplex_mp

# --- IPython ---
from IPython.display import display, Math

# --- Project ---
from max_k_cut import (
    docplex_BQO, docplex_RBQO, docplex_QUBO, docplex_RQUBO, docplex_QUBO_no_constraints,
    tight_qubo_penalty, tight_rqubo_penalty, naive_qubo_penalty, naive_rqubo_penalty,
    interpolated_qubo_penalty, interpolated_rqubo_penalty,
    generate_graph, plot_graph, feasibility_filter, expected_value, sample_std,
    create_dicke_initial_state, create_full_xy_mixer, create_ring_xy_mixer, 
)
from max_k_cut.qaoa import run_qaoa_extract_samples
from scripts.plot_histogram import plot_qaoa_histogram

warnings.filterwarnings('ignore', category=DeprecationWarning)

# --- Version info ---
print("qiskit version:", qiskit.__version__)
print("qiskit aer version:", qiskit_aer.__version__)
print("qiskit algorithms version:", qiskit_algorithms.__version__)
print("qiskit optimization version:", qiskit_optimization.__version__)

# %config InlineBackend.figure_format = 'retina'

In [ ]:
# Problem instance
K = 3
num_nodes = 6
edge_probability = 0.5
weighted = False
weight_range = 1

seed=1
np.random.seed(seed)

G = generate_graph(num_nodes, edge_probability, weighted, weight_range, seed=1)
plot_graph(G)

In [ ]:
# BQO ========================================================
dp_bqo = docplex_BQO(G, K, "Max-K-Cut")

# R-BQO ========================================================
dp_rbqo = docplex_RBQO(G, K, "Max-K-Cut")

# QUBO ========================================================
tight_penalty = tight_qubo_penalty(G, K)
dp_qubo_tight = docplex_QUBO(G, K, tight_penalty,"Max-K-Cut")

naive_penalty = naive_qubo_penalty(G, K)
dp_qubo_naive = docplex_QUBO(G, K, naive_penalty,"Max-K-Cut")

# R-QUBO ========================================================
tight_penalty = tight_rqubo_penalty(G, K)
dp_rqubo_tight = docplex_RQUBO(G, K, tight_penalty, "Max-K-Cut")

naive_penalty = naive_rqubo_penalty(G, K)
dp_rqubo_naive = docplex_RQUBO(G, K, naive_penalty, "Max-K-Cut")

# QUBO with no constraints for mixer========================================================
dp_qubo_no_constraints = docplex_QUBO_no_constraints(G, K, "Max-K-Cut")

In [ ]:
# Solve models and get objective values
objective_values = {
    "BQO": dp_bqo.solve().objective_value,
    "RBQO": dp_rbqo.solve().objective_value,
    "QUBO (Tight)": dp_qubo_tight.solve().objective_value,
    "QUBO (Naive)": dp_qubo_naive.solve().objective_value,
    "RQUBO (Tight)": dp_rqubo_tight.solve().objective_value,
    "RQUBO (Naive)": dp_rqubo_naive.solve().objective_value,
    "QUBO (No Constraints)": dp_qubo_tight.solve().objective_value
}

# Define a tolerance for the comparison
tolerance = 1e-9

# Assert that the objective values are close to each other
bqo_value = objective_values["BQO"]
for label, value in objective_values.items():
    assert math.isclose(bqo_value, value, abs_tol=tolerance), f"{label} value differs, {bqo_value} != {value}"

print(f"All objective values are close to {bqo_value} within the specified tolerance.")

MAX_FVAL = bqo_value

In [ ]:
# parameters
# backend = AerSimulator()
# sampler = Sampler()
# shots = 5000
# sampler.set_options(shots=shots, backend=backend)
# optimizer = COBYLA()
# reps = 3
# initial_point = np.zeros(2 * reps)

# Get the noise model and coupling map from the backend
backend = FakeBrisbane()
backend = AerSimulator.from_backend(backend)
# backend = AerSimulator()
sampler = BackendSampler(backend=backend)
shots = 10000
sampler.set_options(shots=shots)
optimizer = COBYLA()
reps = 4
initial_point = np.random.rand(2 * reps) * np.pi / 2 # initial point for the optimizer


#==============================================================
# regular uniform hadamard initial state, single qubit x mixer
vanilla_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps, 
    initial_point=initial_point,
    )
# create minimum eigen optimizer based on solver used
vanilla_qaoa_optimizer = MinimumEigenOptimizer(vanilla_qaoa)


#==============================================================
# use with qubo model for equality constraints
# dicke initial state, single qubit x mixer
init_qc = create_dicke_initial_state(num_nodes, K)
dicke_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps,
    initial_state=init_qc,
    initial_point=initial_point,
    )
# create minimum eigen optimizer based on solver used
dicke_qaoa_optimizer = MinimumEigenOptimizer(dicke_qaoa)


#==============================================================
# dicke initial state, xy mixer
init_qc = create_dicke_initial_state(num_nodes, K)
beta = Parameter("β")
mixer = create_ring_xy_mixer(num_nodes, K, beta)
# mixer = create_full_xy_mixer(num_nodes, K, beta)

mixer_qaoa = QAOA(
    sampler=sampler, 
    optimizer=optimizer, 
    reps=reps,
    initial_state=init_qc,
    mixer=mixer,
    initial_point=initial_point,
    )

# create minimum eigen optimizer based on solver used
mixer_qaoa_optimizer = MinimumEigenOptimizer(mixer_qaoa)

# Plot all samples

In [ ]:
# Run QAOA to extract raw samples for all models.
# penalty only
penalty_results_dict = run_qaoa_extract_samples(
    [dp_bqo, dp_rbqo, dp_qubo_tight, dp_qubo_naive, dp_rqubo_tight, dp_rqubo_naive],
    ["BQO", "RBQO", "QUBO (Tight)", "QUBO (Naive)", "RQUBO (Tight)", "RQUBO (Naive)"],
    optimizer=vanilla_qaoa_optimizer,
)

# penalty enforcement with dicke initial state
dicke_penalty_results_dict = run_qaoa_extract_samples(
    [dp_qubo_tight, dp_qubo_naive],
    ["QUBO (Tight)", "QUBO (Naive)"],
    optimizer=dicke_qaoa_optimizer,
)

# mixer enforcement
mixer_results_dict = run_qaoa_extract_samples(
    [dp_qubo_no_constraints],
    ["QUBO (No Penalty)"],
    optimizer=mixer_qaoa_optimizer,
)

# penalty enforcement with dicke initial state and xy mixer
penalty_mixer_results_dict = run_qaoa_extract_samples(
    [dp_qubo_tight, dp_qubo_naive],
    ["QUBO (Tight)", "QUBO (Naive)"],
    optimizer=mixer_qaoa_optimizer,
)


# Plots of all samples (feasible and infeasible):

In [ ]:
# Plot the histogram with default settings (no filtering, no re-penalization).
# plot_qaoa_histogram(penalty_results_dict, 'BQO', 'RBQO')
print("penalty only")
plot_qaoa_histogram(penalty_results_dict, 'QUBO (Tight)', 'QUBO (Naive)', 'RQUBO (Tight)', 'RQUBO (Naive)')

In [ ]:
# Plot the histogram with default settings (no filtering, no re-penalization).
print("penalty enforcement + dicke initial state")
plot_qaoa_histogram(dicke_penalty_results_dict, 'QUBO (Tight)', 'QUBO (Naive)')


In [ ]:
print("xy mixer enforcement")
plot_qaoa_histogram(mixer_results_dict, 'QUBO (No Penalty)')

In [ ]:
print("penalty enforcement + xy mixer")
plot_qaoa_histogram(penalty_mixer_results_dict, 'QUBO (Tight)', 'QUBO (Naive)')

# Plot of feasible samples only:

In [ ]:
# Plot the histogram while filtering out infeasible solutions and renormalizing probabilities.
print("penalty only")
plot_qaoa_histogram(penalty_results_dict, 
                    'QUBO (Tight)', 'QUBO (Naive)', 'RQUBO (Tight)', 'RQUBO (Naive)',
                    Graph=G, Partitions=K, filter_infeasible=True)

print("penalty enforcement + dicke initial state")
plot_qaoa_histogram(dicke_penalty_results_dict, 
                    'QUBO (Tight)', 'QUBO (Naive)',
                    Graph=G, Partitions=K, filter_infeasible=True)

print("xy mixer enforcement")
plot_qaoa_histogram(mixer_results_dict, 
                    'QUBO (No Penalty)',
                    Graph=G, Partitions=K, filter_infeasible=True)

print("penalty enforcement + xy mixer")
plot_qaoa_histogram(penalty_mixer_results_dict, 
                    'QUBO (Tight)', 'QUBO (Naive)',
                    Graph=G, Partitions=K, filter_infeasible=True)


In [ ]:
t_list = np.linspace(0, 1, 5)

metrics = {
    "qubo_ratios": [],
    "rqubo_ratios": [],
    "qubo_ratio_stds": [],
    "rqubo_ratio_stds": [],
    "qubo_feas_probs": [],
    "rqubo_feas_probs": [],
    "dicke_qubo_ratios": [],
    "dicke_qubo_ratio_stds": [],
    "dicke_qubo_feas_probs": [],
    "penalty_mixer_qubo_ratios": [],
    "penalty_mixer_qubo_ratio_stds": [],
    "penalty_mixer_qubo_feas_probs": [],
}


def record_metrics(samples, label, ratio_key, std_key, feas_key):
    filtered, feas_prob = feasibility_filter(G, K, samples, label)
    exp_val = expected_value(filtered)
    metrics[ratio_key].append(exp_val)
    metrics[std_key].append(sample_std(filtered, exp_val))
    metrics[feas_key].append(feas_prob)


# --- XY Mixer QAOA (no penalty, t-independent): run once ---
print("  XY Mixer QAOA (t-independent)...")
with contextlib.redirect_stdout(io.StringIO()):
    dp_qubo_no_constraints = docplex_QUBO_no_constraints(G, K, "Max-K-Cut")
    mixer_results = run_qaoa_extract_samples(
        [dp_qubo_no_constraints],
        ["XY Mixer"],
        optimizer=mixer_qaoa_optimizer,
    )

xy_samples = mixer_results["XY Mixer"]["samples"]
xy_filtered, xy_feas_prob = feasibility_filter(G, K, xy_samples, "QUBO (XY Mixer)")
xy_mixer_ratio = expected_value(xy_filtered)
xy_mixer_ratio_std = sample_std(xy_filtered, xy_mixer_ratio)

for t in t_list:
    print(f"  t = {t}")
    with contextlib.redirect_stdout(io.StringIO()):
        dp_qubo_interp = docplex_QUBO(G, K, interpolated_qubo_penalty(G, K, t), "Max-K-Cut")
        dp_rqubo_interp = docplex_RQUBO(G, K, interpolated_rqubo_penalty(G, K, t), "Max-K-Cut")

        penalty_results = run_qaoa_extract_samples(
            [dp_qubo_interp, dp_rqubo_interp],
            ["QUBO (Interpolated)", "RQUBO (Interpolated)"],
            optimizer=vanilla_qaoa_optimizer,
        )
        dicke_results = run_qaoa_extract_samples(
            [dp_qubo_interp],
            ["QUBO (Interpolated)"],
            optimizer=dicke_qaoa_optimizer,
        )
        penalty_mixer_results = run_qaoa_extract_samples(
            [dp_qubo_interp],
            ["QUBO (Interpolated)"],
            optimizer=mixer_qaoa_optimizer,
        )

    qubo_samples = penalty_results["QUBO (Interpolated)"]["samples"]
    rqubo_samples = penalty_results["RQUBO (Interpolated)"]["samples"]
    dicke_qubo_samples = dicke_results["QUBO (Interpolated)"]["samples"]
    penalty_mixer_qubo_samples = penalty_mixer_results["QUBO (Interpolated)"]["samples"]

    record_metrics(
        qubo_samples,
        "QUBO (Interpolated)",
        "qubo_ratios",
        "qubo_ratio_stds",
        "qubo_feas_probs",
    )
    record_metrics(
        rqubo_samples,
        "RQUBO (Interpolated)",
        "rqubo_ratios",
        "rqubo_ratio_stds",
        "rqubo_feas_probs",
    )
    record_metrics(
        dicke_qubo_samples,
        "QUBO (Interpolated)",
        "dicke_qubo_ratios",
        "dicke_qubo_ratio_stds",
        "dicke_qubo_feas_probs",
    )
    record_metrics(
        penalty_mixer_qubo_samples,
        "QUBO (Interpolated)",
        "penalty_mixer_qubo_ratios",
        "penalty_mixer_qubo_ratio_stds",
        "penalty_mixer_qubo_feas_probs",
    )

data = {
    **metrics,
    "xy_mixer_ratio": xy_mixer_ratio,
    "xy_mixer_ratio_std": xy_mixer_ratio_std,
    "xy_mixer_feas_prob": xy_feas_prob,
}

print("Data collection complete.")

In [ ]:
# Custom colors:
qubo_color = "cornflowerblue"
rqubo_color = "orange"
dicke_qubo_color = "green"
penalty_mixer_color = "darkviolet"
xy_mixer_color = "red"

# Create a 2-row figure: top row for expected ratio, bottom row for feasibility probability.
fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True, squeeze=False)


d = data
qubo_ratios = d["qubo_ratios"]
rqubo_ratios = d["rqubo_ratios"]
qubo_ratio_stds = d["qubo_ratio_stds"]
rqubo_ratio_stds = d["rqubo_ratio_stds"]
qubo_feas_probs = d["qubo_feas_probs"]
rqubo_feas_probs = d["rqubo_feas_probs"]
dicke_qubo_ratios = d["dicke_qubo_ratios"]
dicke_qubo_ratio_stds = d["dicke_qubo_ratio_stds"]
dicke_qubo_feas_probs = d["dicke_qubo_feas_probs"]
penalty_mixer_qubo_ratios = d["penalty_mixer_qubo_ratios"]
penalty_mixer_qubo_ratio_stds = d["penalty_mixer_qubo_ratio_stds"]
penalty_mixer_qubo_feas_probs = d["penalty_mixer_qubo_feas_probs"]
xy_mixer_ratio = d["xy_mixer_ratio"]
xy_mixer_feas_prob = d["xy_mixer_feas_prob"]

# -----------------------------
# Top row: Expected Approximation Ratio
# -----------------------------
ax_top = axs[0, 0]
ax_top.plot(t_list, qubo_ratios, label="Penalty QUBO", color=qubo_color, linestyle='--')
ax_top.fill_between(t_list,
                    np.array(qubo_ratios) - np.array(qubo_ratio_stds),
                    np.array(qubo_ratios) + np.array(qubo_ratio_stds),
                    color=qubo_color, alpha=0.1)
ax_top.plot(t_list, rqubo_ratios, label="Penalty RQUBO", color=rqubo_color, linestyle='--')
ax_top.fill_between(t_list,
                    np.array(rqubo_ratios) - np.array(rqubo_ratio_stds),
                    np.array(rqubo_ratios) + np.array(rqubo_ratio_stds),
                    color=rqubo_color, alpha=0.1)
ax_top.plot(t_list, dicke_qubo_ratios, label="Dicke QUBO", color=dicke_qubo_color, linestyle='-')
ax_top.fill_between(t_list,
                    np.array(dicke_qubo_ratios) - np.array(dicke_qubo_ratio_stds),
                    np.array(dicke_qubo_ratios) + np.array(dicke_qubo_ratio_stds),
                    color=dicke_qubo_color, alpha=0.1)
ax_top.plot(t_list, penalty_mixer_qubo_ratios, label="Penalty+Mixer QUBO", color=penalty_mixer_color, linestyle='-.')
ax_top.fill_between(t_list,
                    np.array(penalty_mixer_qubo_ratios) - np.array(penalty_mixer_qubo_ratio_stds),
                    np.array(penalty_mixer_qubo_ratios) + np.array(penalty_mixer_qubo_ratio_stds),
                    color=penalty_mixer_color, alpha=0.1)
ax_top.axhline(y=xy_mixer_ratio, color=xy_mixer_color, linestyle=':', linewidth=2, label="XY Mixer")
ax_top.set_ylim(0, 1)
ax_top.set_ylabel('Approximation Ratio')
ax_top.legend()

# -----------------------------
# Bottom row: Feasibility Probability
# -----------------------------
ax_bottom = axs[1, 0]
ax_bottom.plot(t_list, qubo_feas_probs, label="Penalty QUBO", color=qubo_color, linestyle='--', marker='o')
ax_bottom.plot(t_list, rqubo_feas_probs, label="Penalty RQUBO", color=rqubo_color, linestyle='--', marker='o')
ax_bottom.plot(t_list, dicke_qubo_feas_probs, label="Dicke QUBO", color=dicke_qubo_color, linestyle='-', marker='s')
ax_bottom.plot(t_list, penalty_mixer_qubo_feas_probs, label="Penalty+Mixer QUBO", color=penalty_mixer_color, linestyle='-.', marker='^')
ax_bottom.axhline(y=xy_mixer_feas_prob, color=xy_mixer_color, linestyle=':', linewidth=2, label="XY Mixer")
ax_bottom.set_ylim(0, 1.05)
ax_bottom.set_xlabel('t')
ax_bottom.set_ylabel('Feasibility Probability')
ax_bottom.legend()

plt.tight_layout()
plt.show()

In [ ]:
print(dicke_qubo_ratios)